# Análisis de Modelos de Regresión: Producción de Arroz

El objetivo de este análisis es evaluar el rendimiento de diferentes modelos de regresión (Lineal, Polinomial y técnicas de Regularización) al predecir la **Producción (qq)** de arroz en función de la **Superficie (Ha)** cultivada.

Los datos provienen de la campaña agrícola y presentan una alta variabilidad en los tamaños de las parcelas comunitarias.

## 1. Evaluación del Modelo de Regresión Lineal Simple

Los resultados iniciales del modelo lineal arrojaron las siguientes métricas:
* **R²:** 0.7663
* **MAPE:** 110.43%
* **RMSE:** 19427.76

**Análisis:** 
Aunque el coeficiente de determinación (R² de 0.76) sugiere que el modelo explica una buena porción de la varianza, el error porcentual (MAPE superior al 110%) indica que las predicciones fallan drásticamente en la práctica. 

Al observar la gráfica, el problema es evidente: existe un **valor atípico extremo (outlier)** con más de 8000 Ha y 800,000 qq. El modelo lineal está sesgando su pendiente para intentar reducir el error en este único punto gigante, sacrificando por completo la precisión en el 99% de los datos restantes que se agrupan en superficies menores a 2000 Ha.

<img src="../03_imagenes/imagen_lineal.png" width="700" />

## 2. Regresión Polinomial y Regularización (Ridge, Lasso, Elastic Net)

Para intentar mejorar el ajuste, se aplicó una regresión polinomial y técnicas de regularización, obteniendo:
* **R² (Polinomial, Lasso, Elastic Net):** ~0.819
* **R² (Ridge):** 0.807

**Análisis:**
Matemáticamente, el R² mejoró a casi 0.82. Sin embargo, al analizar la visualización geométrica de las curvas, notamos un claro **sobreajuste (overfitting)** hacia el valor atípico. Las curvas se flexionan de manera irreal en los valores altos de superficie solo para "alcanzar" el dato extremo. 

Esto significa que el modelo está memorizando la anomalía en lugar de aprender el comportamiento agrícola general (rendimiento promedio por hectárea).

<img src="../03_imagenes/imagen_polinomial.png" width="700" />

# Análisis de Regresión: Predicción de Producción de Arroz

Este cuaderno documenta el análisis de los modelos de regresión paramétricos (Lineal y Polinomial) aplicados al dataset de producción de arroz, utilizando la Superficie (Ha) como variable predictora.

## 1. Regresión Lineal Simple

**Métricas obtenidas en consola:**
* **R²:** 0.7664 (76.64%)
* **MAPE:** 110.44%
* **MAE:** 2136.91 qq
* **RMSE:** 19427.76 qq

**Análisis del comportamiento:**
El modelo lineal logró un R² del 76.64%, lo cual indica una correlación fuerte inicial. Sin embargo, las métricas de error revelan un problema crítico con la distribución de los datos. El MAPE supera el 100%, lo que significa que el modelo se equivoca, en promedio, por más del doble del valor real en los pequeños productores. Adicionalmente, la enorme brecha entre el MAE (~2k) y el RMSE (~19k) es un indicador matemático clásico de la presencia de **valores atípicos extremos (outliers)**.

Al observar la gráfica de dispersión, esto se hace evidente: la inmensa mayoría de los productores agrícolas se concentran cerca del origen (menos de 1,000 hectáreas)[cite: 5]. Sin embargo, existe un mega-productor que supera las 8,000 hectáreas y produce más de 800,000 quintales[cite: 5]. La línea de predicción azul, forzada a ser recta, intenta mediar entre ambos extremos, fallando drásticamente en el extremo superior, donde subestima la producción prediciendo apenas 400,000 qq en lugar de los 800,000 reales[cite: 5].

<img src="../03_imagenes/01_regresion_lineal.png" width="700" />

---

## 2. Regresión Polinomial y Regularización

Para capturar la aceleración en la producción a mayor escala, incrementamos la flexibilidad geométrica del modelo a un polinomio de grado 4 y aplicamos penalizaciones para evitar el sobreajuste.

**Métricas obtenidas en consola:**
* **Lineal:** MSE = 4.11e+07 | R² = 0.4155
* **Polinomial (Grado 4):** MSE = 1.27e+07 | R² = 0.8193
* **Lasso (L1):** MSE = 1.27e+07 | R² = 0.8193
* **Elastic Net:** MSE = 1.27e+07 | R² = 0.8195
* **Ridge (L2):** MSE = 1.35e+07 | R² = 0.8079

**Análisis del comportamiento:**
El salto en precisión es notable: los modelos no lineales suben el R² hasta casi un 82%, reduciendo el error cuadrático masivamente respecto al modelo lineal básico de esta ejecución. 

La gráfica comparativa explica el porqué: la curva polinomial (línea morada) y la de Lasso (línea verde, que se superpone a la polinomial) logran doblarse perfectamente para "alcanzar" el valor atípico de los 800,000 qq en las 8,500 hectáreas[cite: 6]. 
* **Ridge (línea azul):** Al penalizar los coeficientes grandes, es el modelo más "rígido" de los no lineales. Crea una curva más suave que no logra tocar el pico máximo, priorizando no sobreajustarse[cite: 6].
* **Elastic Net (línea naranja):** Ofrece un término medio interesante, acelerando antes que Ridge, pero sin la curvatura extrema del polinomio puro[cite: 6].

<img src="../03_imagenes/02_regresion_polinomial.png" width="800" />

### Conclusión Técnica
Para este conjunto de datos específico, los modelos curvos (Polinomial/Elastic Net) representan mejor la relación matemática entre las hectáreas y la producción. No obstante, existe un gran "vacío" de datos entre las 3,000 y 8,000 hectáreas[cite: 6]. El modelo asume el comportamiento en esa zona basándose en un solo punto, lo cual es un riesgo operativo si se incorporan nuevos productores de tamaño medio en el futuro.